# 💻 Hands-On Lab Part 1 — Programmatic Access (45 Minutes)

In this session, we leave the web browsers behind and use Python to query a live cloud registry. Instead of wasting time downloading massive files to our local machines, we will use modern SpatioTemporal Asset Catalog (**STAC**) APIs and Cloud-Optimized GeoTIFFs (**COGs**) to query metadata and stream only the specific pixel layers we need. Furthermore, we will use the GoogleEarthEngine as another option.

---

## 📦 Step 1: Install & Import Core Dependencies

Before running our pipeline, we need to load libraries capable of interacting with modern cloud datacubes. We will use `pystac_client` to communicate with the registry, `planetary_computer` for cloud access keys, and `shapely` to manage our field boundaries.

* **Action Required:** Run the code block below to initialize your environment.

In [1]:
# Uncomment the line below if you need to install the dependencies in your environment
# !pip install pystac-client pystac shapely planetary-computer requests matplotlib

import json
from pystac_client import Client
import planetary_computer as pc
from shapely.geometry import shape, mapping
import matplotlib.pyplot as plt

print("📦 Libraries imported successfully! Ready to connect to the datacube.")


Bad key keymap.all_axes in file matplotlibrc, line 398 ('keymap.all_axes : a                 # enable all axes')
You probably need to get an updated matplotlibrc file from
https://github.com/matplotlib/matplotlib/blob/v3.10.9/lib/matplotlib/mpl-data/matplotlibrc
or from the matplotlib source distribution


📦 Libraries imported successfully! Ready to connect to the datacube.


## 🗺️ Step 2: Define the Agricultural Area of Interest (AOI)

Let's define our boundary box. This is passed to the API query as a GeoJSON coordinate polygon.

* **Action Required:** Execute this cell to lock in our target spatial boundary.
* What are the coordinates for an area near Rostock?

In [2]:
# Define a bounding box around a high-production agricultural region
# Format: [min_longitude, min_latitude, max_longitude, max_latitude]
aoi_bbox = [13.15, 52.35, 13.18, 52.38] 

# Convert the bounding box into a standard GeoJSON geometry dictionary
aoi_geometry = {
    "type": "Polygon",
    "coordinates": [[
        [aoi_bbox[0], aoi_bbox[1]],
        [aoi_bbox[0], aoi_bbox[3]],
        [aoi_bbox[2], aoi_bbox[3]],
        [aoi_bbox[2], aoi_bbox[1]],
        [aoi_bbox[0], aoi_bbox[1]]
    ]]
}

print("Farm boundary geometry locked in. Ready to query.")

Farm boundary geometry locked in. Ready to query.


## 🔍 Step 3: Querying the STAC API (The Metadata Search)

We will now connect to the Microsoft Planetary Computer STAC API endpoint. We are requesting `sentinel-2-l2a` data (**Level-2A Surface Reflectance**), filtered for a spring growing window, and strictly requesting scenes with **less than 10% cloud cover**.

**https://planetarycomputer.microsoft.com/catalog**

* **Action Required:** Run the cell to fetch matching records from the cloud database.

In [3]:
# Connect to the global cloud catalog endpoint
catalog = Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)

# Execute the programmatic search query
search = catalog.search(
    collections=["sentinel-2-l2a"],
    intersects=aoi_geometry,
    datetime="2025-04-01/2025-04-10", # Target spring growing season
    query={"eo:cloud_cover": {"lt": 10}} # Filter: less than 10% cloud cover
)

# Fetch all matching items found in the cloud registry
items = list(search.get_items())
print(f"📡 API Query Complete! Found {len(items)} cloud-free satellite scenes matching your criteria.")

/root/.local/lib/python3.12/site-packages/pystac_client/item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(


📡 API Query Complete! Found 4 cloud-free satellite scenes matching your criteria.


## 📑 Step 4: Inspecting Asset Metadata

Notice that we haven't downloaded any image layers yet. We are looking exclusively at the text-based metadata catalog returned by our query. Let's inspect the first scene found and view the specific bands available for data streaming.

In [4]:
if len(items) > 0:
    best_item = items[0]
    print(f"📋 Selected Scene ID: {best_item.id}")
    print(f"📅 Acquisition Date/Time: {best_item.datetime}")
    print(f"☁️ Exact Cloud Cover Percent: {best_item.properties['eo:cloud_cover']:.2f}%")
    
    print("\n📦 Available Bands (Assets) inside this single item:")
    # Print the internal names of the assets available to stream
    for asset_key, asset in list(best_item.assets.items())[:10]:
        print(f"  - {asset_key}: {asset.title}")
else:
    print("❌ No items found. Try widening your date range or increasing cloud tolerances.")

📋 Selected Scene ID: S2A_MSIL2A_20250404T101041_R022_T33UUU_20250404T184313
📅 Acquisition Date/Time: 2025-04-04 10:10:41.024000+00:00
☁️ Exact Cloud Cover Percent: 0.03%

📦 Available Bands (Assets) inside this single item:
  - AOT: Aerosol optical thickness (AOT)
  - B01: Band 1 - Coastal aerosol - 60m
  - B02: Band 2 - Blue - 10m
  - B03: Band 3 - Green - 10m
  - B04: Band 4 - Red - 10m
  - B05: Band 5 - Vegetation red edge 1 - 20m
  - B06: Band 6 - Vegetation red edge 2 - 20m
  - B07: Band 7 - Vegetation red edge 3 - 20m
  - B08: Band 8 - NIR - 10m
  - B09: Band 9 - Water vapor - 60m


## 🎨 Step 5: Interactive Custom Band Visualization (Spatially Clipped)

Instead of relying on a pre-rendered, compressed visual snapshot of the entire satellite tile, we will now tap directly into the raw spectral data layers. 

Using **Cloud-Optimized GeoTIFFs (COGs)** and `rioxarray`, our Python notebook will request *only* the specific coordinate bounds matching our farm's Area of Interest (AOI). This keeps data transfers incredibly lightweight and fast.

We have built a dual-control cockpit below that allows you to:
1. **Scrub through the Timeline** to watch the fields change over different weeks.
2. **Switch Band Recipes** to analyze properties invisible to the human eye, such as chlorophyll density via Near-Infrared light.

* **Action Required:** Run the interactive cell below to open your farm's dynamic remote sensing deck.

In [12]:
# 1. Fetch and sign all matching scenes found in our query
s2_items = [pc.sign(item) for item in search.get_items()]
print(f"🎞️ Loaded {len(s2_items)} signed scenes into the time-series array.")
s2_items

🎞️ Loaded 4 signed scenes into the time-series array.


[<Item id=S2A_MSIL2A_20250404T101041_R022_T33UUU_20250404T184313>,
 <Item id=S2A_MSIL2A_20250404T101041_R022_T32UQD_20250404T184313>,
 <Item id=S2C_MSIL2A_20250402T101051_R022_T33UUU_20250402T152314>,
 <Item id=S2C_MSIL2A_20250402T101051_R022_T32UQD_20250402T152314>]

In [13]:
# Ensure ipywidgets is installed for interactive rendering
# !pip install ipywidgets requests Pillow matplotlib

import requests
from PIL import Image
import io
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from datetime import datetime

# 1. Define the interactive function that updates the plot when the slider moves
def browse_satellite_scenes(index):
    if not s2_items:
        print("❌ No items found in the time-series array.")
        return
        
    # Get the item corresponding to the slider's current index
    current_item = s2_items[index]
    
    # Format the timestamp nicely for the plot title
    raw_date = current_item.datetime
    formatted_date = raw_date.strftime("%Y-%m-%d %H:%M:%S") if isinstance(raw_date, datetime) else str(raw_date)[:19]
    cloud_cover = current_item.properties.get("eo:cloud_cover", 0.0)
    
    # Grab the visual preview cloud asset URL
    tci_url = current_item.assets["rendered_preview"].href
    
    try:
        # Stream the image data from the cloud into memory
        response = requests.get(tci_url)
        img = Image.open(io.BytesIO(response.content))
        img_array = np.array(img)
        
        # Render the dynamic frame
        plt.figure(figsize=(10, 10))
        plt.imshow(img_array)
        plt.title(f"📅 Date: {formatted_date} | ☁️ Clouds: {cloud_cover:.2f}%\nID: {current_item.id}", fontsize=12, fontweight='bold')
        plt.axis("off")
        plt.show()
        
    except Exception as e:
        print(f"💥 Failed to stream image frame for index {index}: {e}")

# 2. Construct the interactive slider widget boundary
if s2_items:
    slider = widgets.IntSlider(
        value=0,                  # Start at the first image
        min=0,                    # First item index
        max=len(s2_items) - 1,    # Last item index
        step=1,                   # Step increment
        description='Timeline:',
        continuous_update=False,  # Only render when user releases the slider mouse click
        layout=widgets.Layout(width='600px')
    )

    # Display the linked slider and visualization function
    widgets.interact(browse_satellite_scenes, index=slider)
else:
    print("❌ Cannot create slider: search results are empty. Adjust your cloud or date filters.")

interactive(children=(IntSlider(value=0, continuous_update=False, description='Timeline:', layout=Layout(width…

In [14]:
# !pip install rioxarray stackstac rasterio shapely

import requests
import io
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import rioxarray
from shapely.geometry import shape

# Ensure we have our s2_items loaded from previous steps
s2_items = [pc.sign(item) for item in search.get_items()]

# Convert your aoi_geometry dict from Step 2 into a Shapely geometry object for clipping
# We also assume your coordinates are standard WGS84 (EPSG:4326)
cropping_poly = shape(aoi_geometry)

def render_custom_bands(index, combination):
    if not s2_items:
        print("❌ No scenes available.")
        return
        
    current_item = s2_items[index]
    print(f"📡 Slicing cloud tensors for farm AOI: {current_item.id}")
    
    # Define our band recipe mapping based on user selection
    if combination == "True Color (Natural)":
        band_keys = ["B04", "B03", "B02"]  # Red, Green, Blue
    elif combination == "False Color (Infrared Veg)":
        band_keys = ["B08", "B04", "B03"]  # NIR, Red, Green
    elif combination == "Moisture / Agriculture":
        band_keys = ["B12", "B08", "B04"]  # SWIR, NIR, Red

    try:
        rgb_arrays = []
        for band in band_keys:
            # Open the single band direct Cloud-Optimized GeoTIFF (COG) URL
            band_url = current_item.assets[band].href
            
            # Open the raster lazily without downloading yet
            raster = rioxarray.open_rasterio(band_url)
            
            # ✂️ CRITICAL STEP: Crop the raster down to the geometry coordinates.
            # all_touched=True ensures edge pixels are not dropped.
            # from_disk=True avoids reading the whole image before cutting.
            cropped_raster = raster.rio.clip([cropping_poly], crs="EPSG:4326", from_disk=True)
            
            # Extract the raw 2D array data (removing the extra band dimension)
            band_data = cropped_raster.data[0].astype(float)
            rgb_arrays.append(band_data)
        
        # Stack the three chosen cropped bands into an RGB image matrix
        rgb_stacked = np.stack(rgb_arrays, axis=-1)
        
        # Normalize the cropped array for standard 0.0 - 1.0 matplotlib display
        for i in range(3):
            max_val = np.percentile(rgb_stacked[:,:,i], 98) # Dynamic range adjustment
            min_val = np.percentile(rgb_stacked[:,:,i], 2)
            rgb_stacked[:,:,i] = np.clip((rgb_stacked[:,:,i] - min_val) / (max_val - min_val + 1e-5), 0, 1)

        # Plot the exact cropped representation
        plt.figure(figsize=(10, 10))
        plt.imshow(rgb_stacked)
        plt.title(f"✂️ Cropped Field Boundary | Combo: {combination}\n📅 Date: {str(current_item.datetime)[:16]}", fontsize=12)
        plt.axis("off")
        plt.show()
        
    except Exception as e:
        print(f"💥 Network matrix extraction failure: {e}")
        print("💡 Hint: If clipping fails, ensure your bounding box coordinates overlap with this scene's footprint.")

# Build the interactive dual control cockpit
if s2_items:
    widgets.interact(
        render_custom_bands,
        index=widgets.IntSlider(min=0, max=len(s2_items)-1, step=1, value=0, description='Timeline:'),
        combination=widgets.Dropdown(
            options=["True Color (Natural)", "False Color (Infrared Veg)", "Moisture / Agriculture"],
            value="True Color (Natural)",
            description="Band Recipe:"
        )
    )

interactive(children=(IntSlider(value=0, description='Timeline:', max=3), Dropdown(description='Band Recipe:',…

## 🛠️ Student Hands-On Observations

Now that the foundational code runs successfully, apply your knowledge to make the following adjustments:

1. **Change the Coordinates:** Pick a new location or a known agricultural area using a bounding box tool (like `bboxfinder.com`) and replace the coordinates inside **Step 2**.
2. **Shift the Timeline:** Adjust the `datetime` query filter in **Step 3** to look at the peak summer weeks of a different year (e.g., July 2024).
3. **Analyze the Change:** Re-run the cells. How did the shift in geography and season affect your results?
4. **The Infrared Shift:** Switch the **Band Recipe** menu to **False Color (Infrared Veg)**. Notice how some agricultural fields pop into intense shades of crimson or pink, while roads, rivers, and bare soils remain dull grey or green. What is this telling you about the biological activity in those specific fields?
5. **Scrubbing Time:** Use the **Timeline** slider to move from early spring into late spring. Watch how individual fields shift from bare soils (brown/tan in True Color, grey in False Color) into dense, thriving crop canopies.

> **Student Notes / Observation Counter:**
> * **New Location Coordinates:** > * **Number of Scenes Found:** > * **Observed Cloud Cover Variance:** ---